[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/07_%E5%AE%9F%E8%B7%B5_%E3%82%BF%E3%82%A4%E3%82%BF%E3%83%8B%E3%83%83%E3%82%AF%E5%88%86%E6%9E%90.ipynb)

# 統計3級-07. 実践：タイタニック号データで「生存の分かれ目」を分析

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

**これは4級〜3級の総まとめ**です。Kaggleで有名な「タイタニック号」の乗客データを使い、
*誰が助かりやすかったのか* を、習った道具（**データの種類・代表値・ばらつき・箱ひげ図・ヒストグラム・クロス集計・条件付き確率・相関**）だけで読み解きます。


In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルにあればそれを、無ければ公開URL（Kaggle trainと同じ内容）から読み込む
local = '../data/titanic.csv'
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(local) if os.path.exists(local) else pd.read_csv(url)  # データを読み込む
print('乗客数:', len(df))         # 行数＝乗客の人数
df.head()                        # 先頭5行を確認

### 使うデータ：タイタニック号 乗客891人（Kaggle Titanic と同じ内容）
1行＝乗客1人。`Survived` は **1=生存 / 0=死亡**。

| 列 | 意味 | データの種類 |
|---|---|---|
| Survived | 生存(1)/死亡(0) | 質的（名義） |
| Pclass | 客室の等級 1/2/3 | 質的（順序） |
| Sex | 性別 | 質的（名義） |
| Age | 年齢 | 量的 |
| SibSp | 同乗の兄弟姉妹・配偶者の数 | 量的 |
| Parch | 同乗の親・子の数 | 量的 |
| Fare | 運賃 | 量的 |
| Embarked | 乗船した港 C/Q/S | 質的（名義） |

## 1. データの種類と欠損を確認する

分析の第一歩は「どんな列か」「欠けている値はないか」の確認です。

In [ ]:
print(df.dtypes)            # 各列のデータ型（object=文字列, int/float=数値）
print('\n欠損（空欄）の数:')
print(df.isna().sum())      # Age と Cabin に欠損が多い

> **欠損値（けっそんち）**＝記録が空欄のデータ。`Age` は177人ぶん欠けています。
平均などの計算では pandas が欠損を自動でとばします。年齢で分けるときは `dropna()` を使います。

## 2. 代表値とばらつき（Age と Fare）

年齢と運賃について、平均・中央値・標準偏差・変動係数を出します。

In [ ]:
for col in ['Age', 'Fare']:                      # 年齢と運賃について
    s = df[col]
    print(f'【{col}】平均 {s.mean():.1f} / 中央値 {s.median():.1f} / 標準偏差 {s.std():.1f} / 変動係数 {s.std()/s.mean():.2f}')

**読み取り**
- `Fare` は **平均(32) ≫ 中央値(14.5)**。少数の高額運賃に平均が引っ張られている（右に裾が長い）。
- `Fare` は **変動係数が大きい** ＝ 平均に対してばらつきが激しい（運賃の差が大きい）。
- `Age` は平均と中央値が近く、わりと左右対称。

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))   # 1行2列のグラフ
df.boxplot(column='Age', ax=ax[0]); ax[0].set_title('年齢')   # 年齢の箱ひげ図
df.boxplot(column='Fare', ax=ax[1]); ax[1].set_title('運賃')  # 運賃の箱ひげ図
plt.show()

運賃の箱ひげ図は上に外れ値（高額運賃）がたくさん。**「ふつうの乗客」を表すなら中央値**が向いています。

## 3. ヒストグラムで分布を見る

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(df['Age'].dropna(), bins=20, edgecolor='white')   # 年齢（欠損は除く）
ax[0].set_title('年齢の分布')
ax[1].hist(df['Fare'], bins=30, edgecolor='white')           # 運賃
ax[1].set_title('運賃の分布（右に裾）')
plt.show()

## 4. 全体の生存率（確率の基礎）

`Survived` は 0/1 なので、**平均＝生存した人の割合＝生存率**になります。

In [ ]:
print('全体の生存率:', round(df['Survived'].mean(), 3))   # 生存者数 ÷ 全員
print('→ およそ38%が生存。これを基準に、グループ別で比べる')

## 5. クロス集計と条件付き確率 ここが3級の山場

「性別によって生存率はどれだけ違う？」をクロス集計＋行比率で見ます。
行比率は **P(生存 | そのグループ)** ＝ 条件付き確率そのものです。

In [ ]:
ct = pd.crosstab(df['Sex'], df['Survived'])     # 性別×生存 の人数表
print(ct)
print('\n行ごとの割合 = P(生存 | 性別):')
print(pd.crosstab(df['Sex'], df['Survived'], normalize='index').round(3))

**読み取り**：P(生存 | 女性) ≈ **0.74**、P(生存 | 男性) ≈ **0.19**。
全体の0.38と比べ、女性で大きく上がり男性で下がる → **性別と生存に強い関連**（“women and children first”）。

### モザイク図でクロス集計を見る

質的な2変数（性別×生存）の関係は、面積で割合を表す**モザイク図**で視覚化できます。箱の横幅が人数比、縦の分割がその中での生存割合です（統計3級の2変数グラフ）。

In [ ]:
from statsmodels.graphics.mosaicplot import mosaic
fig, _ = mosaic(df, ['Sex','Survived'], gap=0.02, title='性別×生存 のモザイク図')
plt.show()
print('横幅=性別の人数比、縦の分割=その性別での生存(1)/死亡(0)の割合')

In [ ]:
print('等級別の生存率 P(生存 | Pclass):')
print(df.groupby('Pclass')['Survived'].mean().round(3))   # 等級ごとの生存率
df.groupby('Pclass')['Survived'].mean().plot(kind='bar', title='等級別の生存率')
plt.ylabel('生存率'); plt.xticks(rotation=0); plt.show()

1等 0.63 → 2等 0.47 → 3等 0.24。**等級が上がるほど生存率が高い**。

### 性別×等級の合わせ技
2つの条件を重ねると、もっとはっきり見えます。

In [ ]:
# 性別と等級の組み合わせごとの生存率
pd.crosstab([df['Sex'], df['Pclass']], df['Survived'], normalize='index').round(2)

1等の女性はほぼ全員生存、3等の男性は最も低い、と読み取れます。

## 6. 相関の入口（数値どうしの関係）

等級(Pclass)と運賃(Fare)の関係を相関係数で見ます。

In [ ]:
# Pclassは順序尺度だが、数値として相関を見る（1等=1, 3等=3）
print('Pclass と Fare の相関係数:', round(df['Pclass'].corr(df['Fare']), 3))

df['家族人数'] = df['SibSp'] + df['Parch'] + 1     # 本人＋同乗家族＝家族人数
print('家族人数 と Fare の相関係数:', round(df['家族人数'].corr(df['Fare']), 3))
plt.scatter(df['家族人数'], df['Fare'], alpha=0.3)  # 散布図
plt.xlabel('家族人数'); plt.ylabel('運賃'); plt.title('家族人数と運賃'); plt.show()

- `Pclass × Fare` は **負の相関**（等級の数字が小さい＝上等ほど運賃が高い）。
- `家族人数 × Fare` は **やや正**（大人数はまとめて高額になりがち）。

## まとめ

- 助かりやすさ：**女性 > 男性**、**1等 > 2等 > 3等**（年齢が低い子どもも有利）。
- `Fare` は外れ値だらけ → **中央値・箱ひげ図**が役立つ。
- ここで見たのは「差が**ありそう**」まで。**その差が偶然でないか**の確認（カイ二乗検定・t検定）は **2級** で学びます。

---
## 練習問題（3級まで）

**問1.** `Embarked`（乗船港 C/Q/S）ごとの生存率を、クロス集計の行比率で求めよう。最も高い港は？

**問2.** 「子ども(16歳未満)」と「大人」に分けて生存率を比べよう。
ヒント：`df['子ども'] = df['Age'] < 16` を作って `groupby`。

**問3.** 運賃が中央値以上のグループと未満のグループで、生存率を比べよう。

**問4.** `家族人数` と `Age` の相関係数を求めよう。関係は強い？弱い？

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


In [ ]:
# 問4


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[07_実践_タイタニック分析 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/07_%E5%AE%9F%E8%B7%B5_%E3%82%BF%E3%82%A4%E3%82%BF%E3%83%8B%E3%83%83%E3%82%AF%E5%88%86%E6%9E%90.md)**

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""       # ← あなたの名前（例: 山田）
LOG_URL = ""    # ← 記録用URL（配布時に設定）
LOG_TOKEN = ""  # ← 記録用トークン（配布時に設定）
NOTEBOOK = "03_統計_3級/07_実践_タイタニック分析"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")